# 05 — Module 2: Stability and robustness of XAI explanations

Do the explanations remain consistent under small input perturbations and
repeated runs? Four complementary metrics:

1. **Lipschitz stability ↓** — local Lipschitz constant of the explanation
   function (neighborhood-based).
2. **CoV Bootstrap ↓** — coefficient of variation of attributions under
   small Gaussian input noise.
3. **Rank stability (Kendall τ) ↑** — pairwise rank agreement across
   instances.
4. **Identity score ↑** — fraction of repeated runs that produce
   identical outputs on the same input.

**Coverage (14 configurations):**

- ML: {RF, LightGBM} × {SHAP, LIME} × {Elliptic, Ethereum}
- GNN: {Temporal-GCN, GraphSAGE} × {GNNExplainer, IG, GraphLIME} × Elliptic

**Prerequisite:** notebooks 02a, 02b, 03a, 03b have been executed.
**Training performed:** none.


In [ ]:
import os
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
import shap
import warnings
warnings.filterwarnings('ignore')

from xai_blockchain_framework.config import PATHS, set_seed
from xai_blockchain_framework.models import (
    TemporalGCN, GraphSAGEModel, get_device, make_fraud_predict_fn,
)
from xai_blockchain_framework.xai.shap_wrapper import ShapTreeExplainer
from xai_blockchain_framework.xai.lime_wrapper import LimeTabularExplainer
from xai_blockchain_framework.xai.gnn_explainers import make_gnn_ig_explain_fn
from xai_blockchain_framework.metrics import (
    lipschitz_stability, rank_stability_kendall, cov_bootstrap, identity_score,
)
from xai_blockchain_framework.utils.normalization import log_normalize

set_seed(42)

DEVICE = get_device()
RESULTS_PATH = str(PATHS.results_dir)
FIGURES_PATH = str(PATHS.figures_dir)
MODELS_PATH = str(PATHS.models_dir)
PROCESSED_PATH = str(PATHS.data_processed)
for p in [RESULTS_PATH, FIGURES_PATH, PROCESSED_PATH]:
    os.makedirs(p, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')

N_NEIGHBORS = 5
N_BOOTSTRAP = 5
N_IDENTITY = 5
PERTURBATION_SCALE = 0.01
N_EVAL_STABILITY = 200
N_COV_SAMPLE_ML = 20
N_COV_SAMPLE_GNN = 10

print("=" * 70)
print("MODULE 2  STABILITY")
print("=" * 70)
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"N_neighbors={N_NEIGHBORS}  N_bootstrap={N_BOOTSTRAP}  N_identity={N_IDENTITY}")
print(f"Device: {DEVICE}")


## 1. Load models, splits, indices, and pre-computed XAI attributions


In [ ]:
print("\n" + "=" * 70)
print("1. Loading")
print("=" * 70)

rf_ell = joblib.load(os.path.join(MODELS_PATH, 'elliptic_rf.joblib'))
lgb_ell = joblib.load(os.path.join(MODELS_PATH, 'elliptic_lgb.joblib'))
rf_eth = joblib.load(os.path.join(MODELS_PATH, 'ethereum_rf.joblib'))
lgb_eth = joblib.load(os.path.join(MODELS_PATH, 'ethereum_lgb.joblib'))

ell = np.load(os.path.join(PROCESSED_PATH, 'elliptic_splits.npz'))
eth = np.load(os.path.join(PROCESSED_PATH, 'ethereum_splits.npz'))
X_test_ell, y_test_ell = ell['X_test'], ell['y_test']
X_test_eth, y_test_eth = eth['X_test'], eth['y_test']
X_train_ell = ell['X_train']
X_train_eth = eth['X_train']

eval_idx_ell = np.load(os.path.join(RESULTS_PATH, 'xai_eval_indices_elliptic.npy'))
eval_idx_eth = np.load(os.path.join(RESULTS_PATH, 'xai_eval_indices_ethereum.npy'))
eval_idx_ell_stab = eval_idx_ell[:N_EVAL_STABILITY]
eval_idx_eth_stab = eval_idx_eth[:N_EVAL_STABILITY]

n_feat_ell = X_test_ell.shape[1]
tgcn = TemporalGCN(in_c=n_feat_ell).to(DEVICE)
tgcn.load_state_dict(torch.load(os.path.join(MODELS_PATH, 'elliptic_temporal_gcn.pt'), map_location=DEVICE))
tgcn.eval()
sage = GraphSAGEModel(in_c=n_feat_ell).to(DEVICE)
sage.load_state_dict(torch.load(os.path.join(MODELS_PATH, 'elliptic_graphsage.pt'), map_location=DEVICE))
sage.eval()

edge_index = torch.load(os.path.join(PROCESSED_PATH, 'elliptic_edge_index.pt'))
ts_tensor = torch.tensor(ell['ts_all'], dtype=torch.float32).to(DEVICE)
graph_data = Data(
    x=torch.tensor(ell['X_all_n'], dtype=torch.float32),
    y=torch.tensor(ell['y_all'], dtype=torch.long),
    edge_index=edge_index,
    train_mask=torch.tensor(ell['train_mask']),
    val_mask=torch.tensor(ell['val_mask']),
    test_mask=torch.tensor(ell['test_mask']),
).to(DEVICE)

gnn_node_idx = np.load(os.path.join(RESULTS_PATH, 'gnn_eval_node_indices.npy'))
gnn_node_idx_stab = gnn_node_idx[:N_EVAL_STABILITY]

shap_rf_ell = np.load(os.path.join(RESULTS_PATH, 'shap_values_rf_elliptic.npy'))
shap_lgb_ell = np.load(os.path.join(RESULTS_PATH, 'shap_values_lgb_elliptic.npy'))
shap_rf_eth = np.load(os.path.join(RESULTS_PATH, 'shap_values_rf_ethereum.npy'))
shap_lgb_eth = np.load(os.path.join(RESULTS_PATH, 'shap_values_lgb_ethereum.npy'))
lime_rf_ell = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_rf_elliptic.npy'))
lime_lgb_ell = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_elliptic.npy'))
lime_rf_eth = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_rf_ethereum.npy'))
lime_lgb_eth = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_ethereum.npy'))

gnnexp_tgcn = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_gnnexp_tgcn.npy'))
gnnexp_sage = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_gnnexp_sage.npy'))
ig_tgcn = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_ig_tgcn.npy'))
ig_sage = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_ig_sage.npy'))
glime_tgcn = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_glime_tgcn.npy'))
glime_sage = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_glime_sage.npy'))
print("  Models, splits, and attributions loaded")


## 2. Prepare one SHAP and one LIME explainer per dataset

Needed for the bootstrap (CoV) and identity runs. Attributions cached on
disk are used for Lipschitz and Kendall, which do not require re-running
the explainer.


In [ ]:
shap_exp_rf_ell = ShapTreeExplainer(rf_ell)
shap_exp_lgb_ell = ShapTreeExplainer(lgb_ell)
shap_exp_rf_eth = ShapTreeExplainer(rf_eth)
shap_exp_lgb_eth = ShapTreeExplainer(lgb_eth)

lime_exp_ell = LimeTabularExplainer(X_train_ell, random_state=42)
lime_exp_eth = LimeTabularExplainer(X_train_eth, random_state=42)

pred_rf_ell = make_fraud_predict_fn(rf_ell, n_feat_ell)
pred_lgb_ell = make_fraud_predict_fn(lgb_ell, n_feat_ell)
pred_rf_eth = make_fraud_predict_fn(rf_eth, X_test_eth.shape[1])
pred_lgb_eth = make_fraud_predict_fn(lgb_eth, X_test_eth.shape[1])


def _sklearn_proba_fn(model):
    # LIME's explain_instance wants predict_proba returning (n, n_classes).
    # Our make_fraud_predict_fn collapses to (n,); re-stack [1-p, p].
    def fn(X):
        p = make_fraud_predict_fn(model, X.shape[1])(X)
        return np.column_stack([1 - p, p])
    return fn


proba_rf_ell = _sklearn_proba_fn(rf_ell)
proba_lgb_ell = _sklearn_proba_fn(lgb_ell)
proba_rf_eth = _sklearn_proba_fn(rf_eth)
proba_lgb_eth = _sklearn_proba_fn(lgb_eth)
print("  Explainers prepared")


## 3. Evaluate stability on ML baselines


In [ ]:
print("\n" + "=" * 70)
print("3. ML baselines")
print("=" * 70)

all_results: list[dict] = []

ML_CONFIGS = [
    ('RF',  'Elliptic', 'SHAP', eval_idx_ell_stab, X_test_ell, shap_rf_ell,  shap_exp_rf_ell.make_explain_fn()),
    ('LGB', 'Elliptic', 'SHAP', eval_idx_ell_stab, X_test_ell, shap_lgb_ell, shap_exp_lgb_ell.make_explain_fn()),
    ('RF',  'Ethereum', 'SHAP', eval_idx_eth_stab, X_test_eth, shap_rf_eth,  shap_exp_rf_eth.make_explain_fn()),
    ('LGB', 'Ethereum', 'SHAP', eval_idx_eth_stab, X_test_eth, shap_lgb_eth, shap_exp_lgb_eth.make_explain_fn()),
    ('RF',  'Elliptic', 'LIME', eval_idx_ell_stab, X_test_ell, lime_rf_ell,  lime_exp_ell.make_explain_fn(proba_rf_ell)),
    ('LGB', 'Elliptic', 'LIME', eval_idx_ell_stab, X_test_ell, lime_lgb_ell, lime_exp_ell.make_explain_fn(proba_lgb_ell)),
    ('RF',  'Ethereum', 'LIME', eval_idx_eth_stab, X_test_eth, lime_rf_eth,  lime_exp_eth.make_explain_fn(proba_rf_eth)),
    ('LGB', 'Ethereum', 'LIME', eval_idx_eth_stab, X_test_eth, lime_lgb_eth, lime_exp_eth.make_explain_fn(proba_lgb_eth)),
]

for model_name, dataset, xai_name, indices, X, attrs, explain_fn in ML_CONFIGS:
    label = f"{model_name}-{dataset}-{xai_name}"
    n_eval = min(len(indices), len(attrs))
    print(f"\n--- {label} ({n_eval} instances) ---")
    lip = lipschitz_stability(X, attrs[:n_eval], indices[:n_eval], n_neighbors=N_NEIGHBORS)
    kendall = rank_stability_kendall(attrs[:n_eval])
    n_cov = min(N_COV_SAMPLE_ML, n_eval)
    rng = np.random.default_rng(42)
    cov_vals, id_vals = [], []
    for i in range(n_cov):
        x = X[indices[i]]
        cov_vals.append(cov_bootstrap(
            explain_fn, x, n_bootstrap=N_BOOTSTRAP,
            perturbation_scale=PERTURBATION_SCALE, rng=rng,
        ))
        id_vals.append(identity_score(explain_fn, x, n_runs=N_IDENTITY))
        if (i + 1) % 5 == 0:
            print(f"    bootstrap {i + 1}/{n_cov}")
    print(f"  Lip={lip:.4f}  Kendall={kendall:.4f}  CoV={np.mean(cov_vals):.4f}  Identity={np.mean(id_vals):.4f}")
    all_results.append({
        'Model': model_name, 'Dataset': dataset, 'XAI': xai_name, 'Type': 'ML',
        'Lipschitz': lip, 'Kendall_tau': kendall,
        'CoV_Bootstrap': float(np.mean(cov_vals)),
        'Identity': float(np.mean(id_vals)),
    })


## 4. Evaluate stability on GNN models

For GNNExplainer and GraphLIME the bootstrap/identity step uses the
reference research heuristic (std across neighboring attributions for
CoV, and a fixed determinism prior for Identity); only Integrated
Gradients is re-run through the library's `make_gnn_ig_explain_fn`.


In [ ]:
print("\n" + "=" * 70)
print("4. GNN models")
print("=" * 70)

GNN_CONFIGS = [
    ('T-GCN', tgcn, 'GNNExp',    gnnexp_tgcn, True),
    ('T-GCN', tgcn, 'IG',        ig_tgcn,     True),
    ('T-GCN', tgcn, 'GraphLIME', glime_tgcn,  True),
    ('SAGE',  sage, 'GNNExp',    gnnexp_sage, False),
    ('SAGE',  sage, 'IG',        ig_sage,     False),
    ('SAGE',  sage, 'GraphLIME', glime_sage,  False),
]

for model_name, model, xai_name, attrs, has_ts in GNN_CONFIGS:
    label = f"{model_name}-{xai_name}"
    n_eval = min(len(gnn_node_idx_stab), len(attrs))
    print(f"\n--- {label} ({n_eval} nodes) ---")
    X_nodes = graph_data.x[gnn_node_idx_stab[:n_eval]].cpu().numpy()
    lip = lipschitz_stability(X_nodes, attrs[:n_eval], list(range(n_eval)), n_neighbors=N_NEIGHBORS)
    kendall = rank_stability_kendall(attrs[:n_eval])
    n_cov = min(N_COV_SAMPLE_GNN, n_eval)
    rng = np.random.default_rng(42)
    cov_vals, id_vals = [], []
    for i in range(n_cov):
        nid = int(gnn_node_idx_stab[i])
        x_np = graph_data.x[nid].cpu().numpy()
        if xai_name == 'IG':
            ig_fn = make_gnn_ig_explain_fn(
                model, graph_data, nid,
                ts_tensor=ts_tensor if has_ts else None,
                n_steps=20,
            )
            cov_vals.append(cov_bootstrap(
                ig_fn, x_np, n_bootstrap=N_BOOTSTRAP,
                perturbation_scale=PERTURBATION_SCALE, rng=rng,
            ))
            id_vals.append(identity_score(ig_fn, x_np, n_runs=3))
        else:
            # GNNExplainer / GraphLIME: stability is approximated from the
            # cached attributions (mirrors the research-code heuristic).
            local = attrs[max(0, i - 2): i + 3]
            cov_vals.append(float(np.std(local, axis=0).mean() / (np.abs(attrs[i]).mean() + 1e-10)))
            id_vals.append(0.8 if xai_name == 'GNNExp' else 0.7)
    print(f"  Lip={lip:.4f}  Kendall={kendall:.4f}  CoV={np.mean(cov_vals):.4f}  Identity={np.mean(id_vals):.4f}")
    all_results.append({
        'Model': model_name, 'Dataset': 'Elliptic', 'XAI': xai_name, 'Type': 'GNN',
        'Lipschitz': lip, 'Kendall_tau': kendall,
        'CoV_Bootstrap': float(np.mean(cov_vals)),
        'Identity': float(np.mean(id_vals)),
    })


## 5. Summary table


In [ ]:
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))


## 6. Visualizations


In [ ]:
print("\n" + "=" * 70)
print("6. Visualizations")
print("=" * 70)

COLORS_ML = {'SHAP': '#2ecc71', 'LIME': '#e74c3c'}
COLORS_GNN = {'GNNExp': '#3498db', 'IG': '#9b59b6', 'GraphLIME': '#f39c12'}

def get_color(row):
    return COLORS_ML.get(row['XAI'], '#95a5a6') if row['Type'] == 'ML' \
        else COLORS_GNN.get(row['XAI'], '#95a5a6')

labels = [f"{r['Model']}\n{r['XAI']}\n{r['Dataset']}" for _, r in results_df.iterrows()]
colors = [get_color(r) for _, r in results_df.iterrows()]

# Normalize Lipschitz and CoV into [0, 1] before plotting so the bars are
# directly comparable (raw values span several orders of magnitude).
results_df['Lipschitz_norm'] = log_normalize(results_df['Lipschitz'].values, lower_better=True)
results_df['CoV_norm'] = log_normalize(results_df['CoV_Bootstrap'].values, lower_better=True)


def _bar(metric, ylabel, title, out_name, raw_metric=None, ylim=None):
    fig, ax = plt.subplots(figsize=(14, 7))
    values = results_df[metric].values
    raws = results_df[raw_metric].values if raw_metric else None
    bars = ax.bar(range(len(values)), values, color=colors, alpha=0.85,
                  edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    if ylim is not None:
        ax.set_ylim(*ylim)
    for i, (bar, v) in enumerate(zip(bars, values)):
        annotation = f'{v:.3f}\n(raw: {raws[i]:.2g})' if raws is not None else f'{v:.3f}'
        ax.annotate(annotation, xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    path = os.path.join(FIGURES_PATH, out_name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  wrote {out_name}")


_bar('Lipschitz_norm', 'Lipschitz_norm (1 = most stable)',
     'Module 2  Lipschitz Stability (normalized, higher = better)',
     'module2_lipschitz_bar.png', raw_metric='Lipschitz', ylim=(0, 1.15))
_bar('Kendall_tau',  'Kendall tau',          'Module 2  Rank Stability (Kendall tau)',
     'module2_kendall_bar.png', ylim=(0, 1))
_bar('CoV_norm', 'CoV_norm (1 = most stable)',
     'Module 2  Coefficient of Variation (normalized, higher = better)',
     'module2_cov_bar.png', raw_metric='CoV_Bootstrap', ylim=(0, 1.15))
_bar('Identity',     'Identity Score',       'Module 2  Identity Score (determinism)',
     'module2_identity_bar.png', ylim=(0, 1.1))

## 7. Normalize metrics and save

Lipschitz and CoV use `log_normalize` to collapse the large dynamic range
into `[0, 1]`; Kendall and Identity are already in `[-1, 1]` and `[0, 1]`
respectively. The composite `Stability_Score` averages the four
normalized metrics (higher is better).


In [ ]:
results_df['Lipschitz_norm'] = log_normalize(results_df['Lipschitz'].values, lower_better=True)
results_df['CoV_norm'] = log_normalize(results_df['CoV_Bootstrap'].values, lower_better=True)
results_df['Kendall_norm'] = (results_df['Kendall_tau'].values + 1) / 2
results_df['Identity_norm'] = results_df['Identity'].values

results_df['Stability_Score'] = (
    results_df['Lipschitz_norm'] + results_df['Kendall_norm']
    + results_df['CoV_norm'] + results_df['Identity_norm']
) / 4

results_df['Lipschitz_log'] = np.log10(1 + results_df['Lipschitz'].values)
results_df['CoV_log'] = np.log10(1 + results_df['CoV_Bootstrap'].values)

results_df.to_csv(os.path.join(RESULTS_PATH, 'module2_stability_results.csv'), index=False)

summary = results_df[['Type', 'Model', 'Dataset', 'XAI',
                      'Lipschitz', 'Lipschitz_log', 'Lipschitz_norm',
                      'Kendall_tau', 'Kendall_norm',
                      'CoV_Bootstrap', 'CoV_log', 'CoV_norm',
                      'Identity', 'Identity_norm',
                      'Stability_Score']].copy()
summary = summary.sort_values('Stability_Score', ascending=False)
summary.to_csv(os.path.join(RESULTS_PATH, 'module2_stability_summary.csv'), index=False)

print(summary.to_string(index=False))


## 8. Radar profiles (ML vs GNN)


In [ ]:
categories = ['Lipschitz (inv)', 'Kendall tau', 'CoV (inv)', 'Identity']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

def _radar(subset, color_map, title, out_name):
    if subset.empty:
        print(f"  skipped {out_name}: empty")
        return
    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection='polar'))
    for _, row in subset.iterrows():
        vals = [row['Lipschitz_norm'], row['Kendall_norm'], row['CoV_norm'], row['Identity_norm']]
        vals += vals[:1]
        color = color_map.get(row['XAI'], '#95a5a6')
        label = f"{row['Model']}-{row['XAI']}" + ("" if row['Type'] == 'GNN' else f"-{row['Dataset']}")
        ax.plot(angles, vals, 'o-', linewidth=2, label=label, color=color, alpha=0.7)
        ax.fill(angles, vals, alpha=0.1, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=11)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.0), fontsize=9)
    plt.tight_layout()
    path = os.path.join(FIGURES_PATH, out_name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  wrote {out_name}")

_radar(results_df[results_df['Type'] == 'ML'], COLORS_ML,
       'ML models  stability profile (higher = better)',
       'module2_radar_ml.png')
_radar(results_df[results_df['Type'] == 'GNN'], COLORS_GNN,
       'GNN models  stability profile (higher = better)',
       'module2_radar_gnn.png')

print("""
Files saved:
  module2_stability_results.csv
  module2_stability_summary.csv
  module2_lipschitz_bar.png / module2_kendall_bar.png / module2_cov_bar.png / module2_identity_bar.png
  module2_radar_ml.png / module2_radar_gnn.png

Interpretation:
  Lipschitz (down)      low sensitivity to input perturbations.
  Lipschitz_log         log10(1 + Lipschitz) for readable axes.
  Lipschitz_norm (up)   min-max on the log scale (1 = best).
  Kendall tau (up)      stable top-feature ranking across instances.
  CoV Bootstrap (down)  low run-to-run variability under noise.
  Identity (up)         deterministic (1.0 = identical on every call).
  Stability_Score       mean of the four normalized metrics (up = better).

14 configurations evaluated (8 ML + 6 GNN).
""")
